In [1]:
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from pymoo.indicators.hv import HV
from quantum_energy.problem.modeller import RedispatchModel
from quantum_energy.problem.qubo import RedispatchQUBO
from quantum_energy.problem.redispatch_stats import RedispatchStats
from quantum_energy.moo import filter_and_sort_pf

In [2]:
model_type = 'hours=2'

In [3]:
def wes_to_objs(wes):
    objs = []
    for we in wes:
        objs.append(we.energies)
    return np.array(objs)

In [4]:
def objs_to_ref(*objs):
    all_objs = np.vstack(objs)
    return [np.max(all_objs[:,i]) for i in range(all_objs.shape[1])]

In [9]:
norm_stats = {
    "T": [],
    "hv": [],
    "ND": [],
    "overloads": [],
    "costs": [],
    "switches": [],
    "powers": []
}
none_stats = {
    "T": [],
    "hv": [],
    "ND": [],
    "overloads": [],
    "costs": [],
    "switches": [],
    "powers": []
}
with open(f"../data/models/{model_type}/T2.pkl", "rb") as f:
    Rd = pickle.load(f)
Rd_qubo = RedispatchQUBO(Rd)
Rd_stats = RedispatchStats(Rd)
ref_point = [Rd_qubo.bs_load[1], Rd_qubo.bs_cost[1], Rd_qubo.bs_switch[1]]
ranges = [Rd_qubo.bs_load[1] - Rd_qubo.bs_load[0], Rd_qubo.bs_cost[1] - Rd_qubo.bs_cost[0], Rd_qubo.bs_switch[1] - Rd_qubo.bs_switch[0]]
with open(f"../new_results/{model_type}/norm_pf.pkl", 'rb') as f:
    cur_norm_pf = filter_and_sort_pf(Rd_stats, pickle.load(f))
norm_objs = wes_to_objs(cur_norm_pf) * ranges
norm_s = Rd_stats.x_sol_to_stats_over_time(cur_norm_pf[0].x)
for j in range(2):
    norm_stats['T'].append(j+1)
    norm_stats['hv'].append(HV(ref_point=ref_point)(norm_objs))
    norm_stats['ND'].append(len(cur_norm_pf))
    norm_stats['overloads'].append(norm_s['overloads'][j])
    norm_stats['costs'].append(norm_s['costs'][j])
    norm_stats['switches'].append(norm_s['switches'][j])
    norm_stats['powers'].append(norm_s['powers'][j]*100)
with open(f"../new_results/{model_type}/none_pf.pkl", 'rb') as f:
    cur_none_pf = filter_and_sort_pf(Rd_stats, pickle.load(f))
none_objs = wes_to_objs(cur_none_pf)
none_s = Rd_stats.x_sol_to_stats_over_time(cur_none_pf[0].x)
for j in range(2):
    none_stats['T'].append(j+1)
    none_stats['hv'].append(HV(ref_point=ref_point)(none_objs))
    none_stats['ND'].append(len(cur_none_pf))
    none_stats['overloads'].append(none_s['overloads'][j])
    none_stats['costs'].append(none_s['costs'][j])
    none_stats['switches'].append(none_s['switches'][j])  
    none_stats['powers'].append(none_s['powers'][j]*100)

In [10]:
df = pd.concat([pd.DataFrame(norm_stats), pd.DataFrame(none_stats)], keys=["norm", "none"]).reset_index(level=0).rename(columns={"level_0": "method"})
df

,method,T,hv,ND,overloads,costs,switches,powers
0,norm,1,3.266896e+14,30,5,1.445210e+06,0,0.013970
1,norm,2,3.266896e+14,30,4,1.276690e+06,156,-0.013563
0,none,1,2.700502e+14,15,7,1.147317e+06,0,-0.018859
1,none,2,2.700502e+14,15,6,1.032040e+06,194,-0.014899


In [12]:
df_agg.to_csv('./compare-scalers.csv')

In [61]:
none_hvs

[135167049248115.6,
 134728366556098.98,
 127501246678540.23,
 134238798359589.81,
 137351505906774.53,
 137941600263930.62,
 139346749683293.56,
 137776649611391.45,
 134428500436774.14,
 133467619756658.88]

In [19]:
len(filter_and_sort_pf(Rd_stats, tmp_none_pf, 0.01))

50

In [35]:
tmp_obj_1 = wes_to_objs(tmp_norm_pf)
tmp_obj_2 = wes_to_objs(tmp_stan_pf)

In [36]:
tmp_obj_1

array([[0.63140499, 0.35254627, 0.44500284],
       [0.63147368, 0.31083001, 0.27531783],
       [0.63143388, 0.30978569, 0.25375391],
       [0.63021378, 0.35739694, 0.27360211],
       [0.63399348, 0.27634222, 0.10117265],
       [0.63427645, 0.27457655, 0.0562006 ]])

In [37]:
tmp_obj_2

array([[18.71330578, 13.18964461,  8.92933583],
       [18.71259254, 13.20310389,  8.9295675 ],
       [18.71156893, 13.23022825,  8.94434935],
       [18.65446334, 16.09225871, 13.50325393],
       [18.71876634, 12.94696486,  5.5603249 ],
       [18.70855439, 13.09883262,  5.43257822],
       [18.79207115, 12.15488676,  5.50359299],
       [18.7581341 , 12.49870566,  5.53274399],
       [18.86108974, 12.58267644,  0.        ],
       [18.88757022, 12.7452672 ,  0.        ],
       [18.89600059, 12.82656379,  0.        ]])

In [38]:
objs_to_ref(tmp_obj_1, tmp_obj_2)

[18.896000588634248, 16.09225870672882, 13.503253928458875]

In [22]:
len(filter_and_sort_pf(Rd_stats, tmp_norm_pf, 0.01))

6

In [21]:
len(filter_and_sort_pf(Rd_stats, tmp_stan_pf, 0.01))

11